![A car dashboard with lots of new technical features.](images/dashboard.jpg)

You're working for a well-known car manufacturer who is looking at implementing LLMs into vehicles to provide guidance to drivers. You've been asked to experiment with integrating car manuals with an LLM to create a context-aware chatbot. They hope that this context-aware LLM can be hooked up to a text-to-speech software to read the model's response aloud.

As a proof of concept, you'll integrate several pages from a car manual that contains car warning messages and their meanings and recommended actions. This particular manual, stored as an HTML file, `mg-zs-warning-messages.html`, is from an MG ZS automobile, a compact SUV. Armed with your newfound knowledge of LLMs and LangChain, you'll implement Retrieval Augmented Generation (RAG) to create the context-aware chatbot.

**Note: Although we'll be using the OpenAI API in this project, you do not need to specify an API key.**

In [1]:
# Run this cell to install the necessary packages
import subprocess
import pkg_resources

def install_if_needed(package, version):
    '''Function to ensure that the libraries used are consistent to avoid errors.'''
    try:
        pkg = pkg_resources.get_distribution(package)
        if pkg.version != version:
            raise pkg_resources.VersionConflict(pkg, version)
    except (pkg_resources.DistributionNotFound, pkg_resources.VersionConflict):
        subprocess.check_call(["pip", "install", f"{package}=={version}"])

install_if_needed("langchain-core", "0.3.72")
install_if_needed("langchain-openai", "0.3.28")
install_if_needed("langchain-community", "0.3.27")
install_if_needed("unstructured", "0.18.11")
install_if_needed("langchain-chroma", "0.2.5")
install_if_needed("langchain-text-splitters", "0.3.9")
install_if_needed("pydantic", "2.11.9")

In [ ]:
# Import the required packages
import hashlib
import math
import re

from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_chroma import Chroma

In [ ]:
# Load the HTML as LangChain documents, one per warning message
with open("data/mg-zs-warning-messages.html", encoding="utf-8") as html_file:
    soup = BeautifulSoup(html_file, "html.parser")

car_docs = []
for row in soup.find_all("tr"):
    cells = row.find_all("td")
    if len(cells) != 2:
        continue

    warning = " ".join(cells[0].stripped_strings)
    procedure = " ".join(cells[1].stripped_strings)

    if not warning or warning.lower() == "warning message":
        continue

    car_docs.append(
        Document(
            page_content=f"Warning message: {warning}\nProcedure: {procedure}",
            metadata={
                "warning_message": warning,
                "procedure": procedure,
                "source": "data/mg-zs-warning-messages.html",
            },
        )
    )

len(car_docs)

In [ ]:
# Local embeddings keep the notebook runnable without an external API key.
class SimpleHashEmbeddings(Embeddings):
    def __init__(self, dimensions: int = 256):
        self.dimensions = dimensions

    def _embed(self, text: str) -> list[float]:
        tokens = re.findall(r"[a-z0-9]+", text.lower())
        vector = [0.0] * self.dimensions

        for token in tokens:
            index = int(hashlib.md5(token.encode("utf-8")).hexdigest(), 16) % self.dimensions
            vector[index] += 1.0

        norm = math.sqrt(sum(value * value for value in vector)) or 1.0
        return [value / norm for value in vector]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text)

embeddings = SimpleHashEmbeddings()

In [ ]:
# Create vector database
vectorstore = Chroma.from_documents(
    documents=car_docs,
    embedding=embeddings
)

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def answer_from_context(inputs):
    docs = inputs["context"]
    if not docs:
        return "I don't know."

    best_match = docs[0]
    warning = best_match.metadata["warning_message"]
    procedure = best_match.metadata["procedure"]
    return f"{warning}: {procedure}"

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | RunnableLambda(answer_from_context)
)

# User query
query = "The Gasoline Particular Filter Full warning has appeared. What does this mean and what should I do about it?"

# Store answer
answer = rag_chain.invoke(query)

print(answer)